# Phase 2: BERT / RoBERTa Baseline Training

**CSCI E-222 · Spring 2026**

This notebook trains the multi-label BERT and RoBERTa classifiers, tunes per-label thresholds on the validation set, and logs all evaluation metrics. The best checkpoint serves as the latency and performance baseline for Phases 3–5.

**Expected runtime on A100:** 30–60 min per model.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from data.product_dataset import ProductDataset
from models.bert_classifier import BertMultiLabelClassifier, BertTrainer, predict_batch, measure_latency
from eval.metrics import compute_metrics, tune_thresholds, per_label_f1

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid', palette='muted')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Taxonomy & Dataset Stats

In [ ]:
with open('../data/taxonomy.json') as f:
    taxonomy = json.load(f)
LABEL_NAMES = [l['tag'] for l in taxonomy['labels']]
NUM_LABELS = len(LABEL_NAMES)

with open('../data/processed/stats.json') as f:
    stats = json.load(f)

POS_WEIGHTS = stats['per_label_pos_weights']

print(f'Labels: {NUM_LABELS}')
print(f'Train: {stats["n_train"]:,} | Val: {stats["n_val"]:,} | Test: {stats["n_test"]:,}')
print(f'Avg labels per product: {stats["avg_labels_per_product"]}')

## 2. Training Configuration

Change `MODEL_NAME` to `'roberta-base'` to train the RoBERTa variant.

In [ ]:
MODEL_NAME  = 'bert-base-uncased'   # swap to 'roberta-base' for RoBERTa run
BATCH_SIZE  = 32
NUM_EPOCHS  = 5
MAX_LENGTH  = 120
CHECKPOINT  = f'../checkpoints/{MODEL_NAME.replace("/", "-")}'

TRAIN_CONFIG = {
    'learning_rate': 2e-5,
    'weight_decay':  0.01,
    'num_epochs':    NUM_EPOCHS,
    'max_grad_norm': 1.0,
    'batch_size':    BATCH_SIZE,
    'max_length':    MAX_LENGTH,
    'model_name':    MODEL_NAME,
}

print(json.dumps(TRAIN_CONFIG, indent=2))

## 3. DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = ProductDataset('../data/processed/train.parquet', tokenizer, MAX_LENGTH, NUM_LABELS)
val_ds   = ProductDataset('../data/processed/val.parquet',   tokenizer, MAX_LENGTH, NUM_LABELS)
test_ds  = ProductDataset('../data/processed/test.parquet',  tokenizer, MAX_LENGTH, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

## 4. Train

In [ ]:
model = BertMultiLabelClassifier(MODEL_NAME, NUM_LABELS)

trainer = BertTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=TRAIN_CONFIG,
    checkpoint_dir=CHECKPOINT,
    pos_weight=POS_WEIGHTS,
    device=DEVICE,
)

results = trainer.train()
print(f"\nBest Micro-F1: {results['best_micro_f1']:.4f}")

## 5. Loss & F1 Curves

In [ ]:
history = results['history']
epochs  = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(epochs, history['train_loss'], label='Train loss', marker='o')
axes[0].plot(epochs, history['val_loss'],   label='Val loss',   marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCEWithLogitsLoss')
axes[0].set_title(f'{MODEL_NAME} — Loss')
axes[0].legend()

# F1 curves
axes[1].plot(epochs, history['micro_f1'], label='Micro-F1', marker='o')
axes[1].plot(epochs, history['macro_f1'], label='Macro-F1', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].set_title(f'{MODEL_NAME} — Validation F1')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'../data/processed/{MODEL_NAME.replace("/", "-")}_curves.png', dpi=150)
plt.show()

## 6. Evaluate Best Checkpoint on Test Set

In [ ]:
best_model = BertMultiLabelClassifier.from_checkpoint(CHECKPOINT)
best_model.eval().to(DEVICE)

best_thresholds = results['best_thresholds']

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        logits = best_model(input_ids, attention_mask, token_type_ids)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(batch['labels'].numpy())

y_prob = np.vstack(all_probs)
y_true = np.vstack(all_labels)

test_metrics = compute_metrics(y_true, y_prob, best_thresholds)

print('Test set results:')
for k, v in test_metrics.items():
    if k != 'per_label_f1':
        print(f'  {k:<20} {v}')

## 7. Per-Label F1 Bar Chart

In [ ]:
label_f1 = dict(zip(LABEL_NAMES, test_metrics['per_label_f1']))
sorted_items = sorted(label_f1.items(), key=lambda x: x[1])
names, scores = zip(*sorted_items)

fig, ax = plt.subplots(figsize=(8, 9))
bars = ax.barh(names, scores, color='steelblue')
ax.axvline(test_metrics['micro_f1'], color='tomato', linestyle='--', label=f'Micro-F1 = {test_metrics["micro_f1"]:.3f}')
ax.set_xlabel('F1 score')
ax.set_title(f'{MODEL_NAME} — Per-label F1 (test set)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f'../data/processed/{MODEL_NAME.replace("/", "-")}_per_label_f1.png', dpi=150)
plt.show()

## 8. Inference Latency

In [ ]:
latency = measure_latency(best_model, tokenizer, n_samples=200, max_length=MAX_LENGTH, device=DEVICE)

print('Inference latency (single sample, no batching):')
for k, v in latency.items():
    print(f'  {k:<12} {v} ms')

# Save latency results for Phase 3 comparison
with open(f'../data/processed/{MODEL_NAME.replace("/", "-")}_latency.json', 'w') as f:
    json.dump({'model': MODEL_NAME, **latency}, f, indent=2)

## 9. Threshold Report

In [ ]:
threshold_df = pd.DataFrame({
    'label':     LABEL_NAMES,
    'threshold': best_thresholds,
    'test_f1':   test_metrics['per_label_f1'],
}).sort_values('test_f1')

print(threshold_df.to_string(index=False))

print('\nPhase 2 complete.')
print(f'Checkpoint saved to: {CHECKPOINT}')